In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [22]:
df = pd.read_csv("Students_analysis.csv")

print(df.shape)
df.head()

(1000, 12)


,StudentID,Name,Gender,AttendanceRate,StudyHoursPerWeek,PreviousGrade,ExtracurricularActivities,ParentalSupport,FinalGrade,Study Hours,Attendance (%),Online Classes Taken
0,1.0,John,Male,85.0,15.0,78.0,1.0,High,80.0,4.8,59.0,False
1,2.0,Sarah,Female,90.0,20.0,85.0,2.0,Medium,87.0,2.2,70.0,True
2,3.0,Alex,Male,78.0,10.0,65.0,0.0,Low,68.0,4.6,92.0,False
3,4.0,Michael,Male,92.0,25.0,90.0,3.0,High,92.0,2.9,96.0,False
4,5.0,Emma,Female,NaN,18.0,82.0,2.0,Medium,85.0,4.1,97.0,True


In [ ]:
df.info()

In [ ]:
target = "FinalGrade"

features = [
    col for col in df.columns
    if col not in ["FinalGrade", "StudentID", "Name"]
]

X = df[features]
y = df[target]

print("Features:")
print(features)

print("\nTarget:")
print(target)

In [ ]:
mask = y.notna()

X = X.loc[mask]
y = y.loc[mask]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
categorical_columns = [
    col for col in features
    if X[col].dtype == "object"
]

numerical_columns = [
    col for col in features
    if col not in categorical_columns
]

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
svm_preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numerical_columns
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_columns
    )
])

In [ ]:
svm_model = Pipeline([
    ("preprocessor", svm_preprocessor),
    (
        "model",
        SVR(
            kernel="rbf",
            C=100,
            gamma="scale",
            epsilon=0.1
        )
    )
])

In [ ]:
svm_model.fit(X_train, y_train)

In [ ]:
svm_predictions = svm_model.predict(X_test)

print(svm_predictions[:10])

In [ ]:
svm_mae = mean_absolute_error(y_test, svm_predictions)
svm_rmse = np.sqrt(mean_squared_error(y_test, svm_predictions))
svm_r2 = r2_score(y_test, svm_predictions)

print("SVM Results")
print("MAE :", svm_mae)
print("RMSE:", svm_rmse)
print("R2  :", svm_r2)

In [ ]:
rf_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median"),
        numerical_columns
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_columns
    )
])

In [ ]:
rf_model = Pipeline([
    ("preprocessor", rf_preprocessor),
    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
    )
])

In [ ]:
rf_model.fit(X_train, y_train)

In [ ]:
rf_predictions = rf_model.predict(X_test)

print(rf_predictions[:10])

In [ ]:
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest Results")
print("MAE :", rf_mae)
print("RMSE:", rf_rmse)
print("R2  :", rf_r2)

In [ ]:
results = pd.DataFrame({
    "Model": ["SVM", "Random Forest"],
    "MAE": [svm_mae, rf_mae],
    "RMSE": [svm_rmse, rf_rmse],
    "R2": [svm_r2, rf_r2]
})

results

In [ ]:
if rf_r2 > svm_r2:
    print("Random Forest performs better.")
else:
    print("SVM performs better.")